**Instalare PySpark in Colab**

In [20]:
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install -q pyspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [22]:
from google.colab import files
uploaded = files.upload()

Saving coin_USDCoin.csv to coin_USDCoin (1).csv


**Initializare Spark si incarcare date**

In [23]:
spark = SparkSession.builder \
    .appName("USDCoin Analysis") \
    .getOrCreate()

file_path = "coin_USDCoin.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

print("PAS 1: Initializez SparkSession si incarc Fisierul CSV.")
print(f"Numar inregistrari: {df.count()}, Coloane: {df.columns}")


PAS 1: Initializez SparkSession si incarc Fisierul CSV.
Numar inregistrari: 1002, Coloane: ['SNo', 'Name', 'Symbol', 'Date', 'High', 'Low', 'Open', 'Close', 'Volume', 'Marketcap']


**Curatarea datelor**


In [24]:
df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
df = df.dropna()
print("PAS 2: Curatam data.")
df.show()
df.printSchema()

PAS 2: Curatam data.
+---+--------+------+----------+-------+--------+-------+-------+---------+---------------+
|SNo|    Name|Symbol|      Date|   High|     Low|   Open|  Close|   Volume|      Marketcap|
+---+--------+------+----------+-------+--------+-------+-------+---------+---------------+
|  1|USD Coin|  USDC|2018-10-09|1.00833| 1.00003|1.00427|1.00686| 108803.0|            0.0|
|  2|USD Coin|  USDC|2018-10-10|1.01908| 1.00397|1.00494|1.00962| 711783.0|            0.0|
|  3|USD Coin|  USDC|2018-10-11|1.03123| 1.00173|1.00994|1.00978|4177290.0|            0.0|
|  4|USD Coin|  USDC|2018-10-12|1.01871| 1.00114|1.00861|1.01272|1322240.0|            0.0|
|  5|USD Coin|  USDC|2018-10-13|1.02029| 1.00355|1.01445|1.00743| 698507.0|            0.0|
|  6|USD Coin|  USDC|2018-10-14|1.02251| 1.00492|1.00726|1.01631| 982750.0|            0.0|
|  7|USD Coin|  USDC|2018-10-15|1.11373|0.998372| 1.0169| 1.0268|1.06604E7|            0.0|
|  8|USD Coin|  USDC|2018-10-16|1.03551| 1.01899|1.02723|1.

**Agregare cu DataFrame API**

In [25]:
monthly_stats = df.withColumn("Year", year("Date")) \
    .withColumn("Month", month("Date")) \
    .groupBy("Year", "Month") \
    .agg(
        avg("Close").alias("Average_Close"),
        sum("Volume").alias("Total_Volume")
    ).orderBy("Year", "Month")

monthly_stats.show()
print("PAS 3: Agregare lunara realizata cu DataFrame API.")

+----+-----+------------------+--------------------+
|Year|Month|     Average_Close|        Total_Volume|
+----+-----+------------------+--------------------+
|2018|   10|1.0105476521739132|         3.5524943E7|
|2018|   11|1.0125907151823337| 3.662171539774874E8|
|2018|   12|1.0145268469305808| 6.717680147434705E8|
|2019|    1|1.0120390603780642| 6.724316133119994E8|
|2019|    2|1.0134725103628572| 9.256489224572749E8|
|2019|    3|1.0127588813490325| 9.132530341225402E8|
|2019|    4|1.0050269621103665| 1.563438117448276E9|
|2019|    5|1.0028129701066129| 5.512744774092041E9|
|2019|    6|   1.0005787886751| 3.787835892868051E9|
|2019|    7| 1.000926038160516|3.7397778306837206E9|
|2019|    8|1.0008985845309677|4.1009862489423404E9|
|2019|    9|   1.0019283893576| 5.760231702407042E9|
|2019|   10|1.0030883986638064| 6.232813543519031E9|
|2019|   11|1.0047688019508667| 7.694717145627366E9|
|2019|   12|1.0050639325501292| 8.817167457965576E9|
|2020|    1| 1.007602499397355|1.1241648392091

**Agregare cu Spark SQL**

In [26]:
df.createOrReplaceTempView("usdc")
sql_result = spark.sql("""
    SELECT year(Date) as Year, month(Date) as Month,
           AVG(Close) as Avg_Close,
           SUM(Volume) as Total_Volume
    FROM usdc
    GROUP BY year(Date), month(Date)
    ORDER BY year(Date), month(Date)
""")
sql_result.show()
print("PAS 4: Agregare lunara cu Spark SQL.")


+----+-----+------------------+--------------------+
|Year|Month|         Avg_Close|        Total_Volume|
+----+-----+------------------+--------------------+
|2018|   10|1.0105476521739132|         3.5524943E7|
|2018|   11|1.0125907151823337| 3.662171539774874E8|
|2018|   12|1.0145268469305808| 6.717680147434705E8|
|2019|    1|1.0120390603780642| 6.724316133119994E8|
|2019|    2|1.0134725103628572| 9.256489224572749E8|
|2019|    3|1.0127588813490325| 9.132530341225402E8|
|2019|    4|1.0050269621103665| 1.563438117448276E9|
|2019|    5|1.0028129701066129| 5.512744774092041E9|
|2019|    6|   1.0005787886751| 3.787835892868051E9|
|2019|    7| 1.000926038160516|3.7397778306837206E9|
|2019|    8|1.0008985845309677|4.1009862489423404E9|
|2019|    9|   1.0019283893576| 5.760231702407042E9|
|2019|   10|1.0030883986638064| 6.232813543519031E9|
|2019|   11|1.0047688019508667| 7.694717145627366E9|
|2019|   12|1.0050639325501292| 8.817167457965576E9|
|2020|    1| 1.007602499397355|1.1241648392091

**Pregatire date pentru Machine Learning**

In [27]:
features = ["Open", "High", "Low", "Volume", "Marketcap"]
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_ml = assembler.transform(df).select("features", "Close")
print("PAS 5: VectorAssembler a fost aplicat.")
df_ml.select("features", "Close").show(3)


PAS 5: VectorAssembler a fost aplicat.
+--------------------+-------+
|            features|  Close|
+--------------------+-------+
|[1.00427,1.00833,...|1.00686|
|[1.00494,1.01908,...|1.00962|
|[1.00994,1.03123,...|1.00978|
+--------------------+-------+
only showing top 3 rows



**Regresie liniara + evaluare**

In [28]:
lr = LinearRegression(featuresCol="features", labelCol="Close")
lr_model = lr.fit(df_ml)
predictions = lr_model.transform(df_ml)

reg_eval = RegressionEvaluator(labelCol="Close", predictionCol="prediction", metricName="rmse")
rmse = reg_eval.evaluate(predictions)
print(f"PAS 6: Model de regresie antrenat. RMSE: {rmse:.4f}")


PAS 6: Model de regresie antrenat. RMSE: 0.0034


**Clasificare volatilitate + AUC**

In [29]:
df_class = df.withColumn("volatility", when((col("High") - col("Low")) > 0.01, 1).otherwise(0))
assembler_cls = VectorAssembler(inputCols=features, outputCol="features")
df_class = assembler_cls.transform(df_class).select("features", "volatility")
print("PAS 7: Eticheta 'volatility' a fost adaugata.")
df_class.select("features", "volatility").show()

lr_cls = LogisticRegression(featuresCol="features", labelCol="volatility")
model_cls = lr_cls.fit(df_class)
preds = model_cls.transform(df_class)

cls_eval = BinaryClassificationEvaluator(labelCol="volatility")
auc = cls_eval.evaluate(preds)
print(f"PAS 8: Model de clasificare antrenat. AUC: {auc:.4f}")

PAS 7: Eticheta 'volatility' a fost adaugata.
+--------------------+----------+
|            features|volatility|
+--------------------+----------+
|[1.00427,1.00833,...|         0|
|[1.00494,1.01908,...|         1|
|[1.00994,1.03123,...|         1|
|[1.00861,1.01871,...|         1|
|[1.01445,1.02029,...|         1|
|[1.00726,1.02251,...|         1|
|[1.0169,1.11373,0...|         1|
|[1.02723,1.03551,...|         1|
|[1.02404,1.02653,...|         1|
|[1.01466,1.02062,...|         1|
|[1.01475,1.01643,...|         1|
|[1.00749,1.01363,...|         1|
|[1.00682,1.01961,...|         1|
|[1.01231,1.0248,1...|         1|
|[1.00923,1.01796,...|         1|
|[1.00938,1.01209,...|         0|
|[1.01046,1.01369,...|         1|
|[1.00611,1.01552,...|         1|
|[1.01208,1.01489,...|         1|
|[1.00662,1.01496,...|         1|
+--------------------+----------+
only showing top 20 rows

PAS 8: Model de clasificare antrenat. AUC: 1.0000


**Pipeline ML**

In [30]:
pipeline = Pipeline(stages=[assembler, lr])
model_pipeline = pipeline.fit(df)
pipeline_predictions = model_pipeline.transform(df)
pipeline_predictions.select("prediction", "Close").show()
print("PAS 9: Pipeline ML executat cu succes.")


+------------------+-------+
|        prediction|  Close|
+------------------+-------+
|1.0038499841487953|1.00686|
| 1.008671962314031|1.00962|
|1.0128310116258301|1.00978|
|1.0084039177106456|1.01272|
|1.0109176318743307|1.00743|
| 1.010553366738263|1.01631|
| 1.039260805660155| 1.0268|
|1.0234497557602857|1.02143|
|1.0171987832349434|1.01219|
|1.0129183857111492|1.01489|
|1.0106612630174574|1.00698|
|1.0070231530471396|1.00643|
| 1.008983287487037| 1.0129|
|1.0118067768711654| 1.0093|
|1.0103621897417103|1.01059|
|1.0068777251687946|1.01002|
|1.0074537468059948|1.00797|
|1.0076445895231942|1.01163|
|1.0079022210200004|1.00824|
|1.0073173117247518|1.01107|
+------------------+-------+
only showing top 20 rows

PAS 9: Pipeline ML executat cu succes.


**UDF categorizare volum**

In [31]:
def volume_category(volume):
    if volume < 1e6:
        return "low"
    elif volume < 10e6:
        return "medium"
    else:
        return "high"

volume_udf = udf(volume_category, StringType())
df = df.withColumn("VolumeCategory", volume_udf(col("Volume")))
print("PAS 10: UDF aplicat pentru categorizarea volumului.")
df.select("Volume", "VolumeCategory").show()


PAS 10: UDF aplicat pentru categorizarea volumului.
+---------+--------------+
|   Volume|VolumeCategory|
+---------+--------------+
| 108803.0|           low|
| 711783.0|           low|
|4177290.0|        medium|
|1322240.0|        medium|
| 698507.0|           low|
| 982750.0|           low|
|1.06604E7|          high|
|2124080.0|        medium|
|1102740.0|        medium|
|1182450.0|        medium|
| 715078.0|           low|
| 299871.0|           low|
| 509276.0|           low|
|1211160.0|        medium|
|1503840.0|        medium|
| 923940.0|           low|
|1365520.0|        medium|
| 950623.0|           low|
| 712963.0|           low|
| 282707.0|           low|
+---------+--------------+
only showing top 20 rows



**CrossValidator - optimizare hiperparametri**

In [32]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .build()

crossval = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=reg_eval,
    numFolds=3
)

cv_model = crossval.fit(df_ml)
best_model = cv_model.bestModel
best_rmse = reg_eval.evaluate(best_model.transform(df_ml))
print(f"PAS 11: CrossValidator a antrenat model: {best_rmse:.4f}")


PAS 11: CrossValidator a antrenat model: 0.0042


**Oprirea Spark**

In [33]:
spark.stop()